In [1]:
import numpy as np
from aeon.classification.interval_based import RSTSF
from sklearn.metrics import accuracy_score
from tscglue.models import LokyStackerV7, LokyStackerV7SoftFilterRidge, TSCGlue, LokyStackerV9Base, LokyStackerV10Base
from tscglue import utils, data_loader
import polars as pl
from sklearn.metrics import log_loss, roc_auc_score


In [2]:
dataset = "Worms"
# dataset = 'Car'
# dataset = 'HandOutlines'
# dataset = 'Trace'
#dataset = 'SwedishLeaf'
#dataset = 'FordA'
X_train, y_train, X_test, y_test = data_loader.load_fold(dataset, 0)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(181, 1, 900) (181,) (77, 1, 900) (77,)


In [3]:
seed = 26

In [4]:
m = LokyStackerV10Base(random_state=seed, n_jobs=16, keep_features=True, verbose=10, n_repetitions=2)

In [5]:
#model_specs = m.build_model_specs([
#    "multirockethydra-bestk-p-ridgecv", 
#    "quant-etc", 
#    "rdst-p-ridgecv", 
#    "rstsf",
#    "multirockethydra-bestk-p-ridgecv"
#])
#model_specs

In [6]:
m.fit(X_train, y_train)

[0.00s] Starting fit, run_dir=tscglue_runs/7900e4e36db04874, n_jobs=16
[0.00s] Saved X and y to disk in 0.00s
[5.07s] Computed multirocket_s_292867638 features (181, 49728) (34.34 MB) in 0.9346s
[7.26s] Computed hydra_s_732432917 features torch.Size([181, 7168]) (4.95 MB) in 2.1855s
[9.48s] Computed quant features (181, 7987) (5.51 MB) in 2.2243s
[15.82s] Computed rdst_s_121263262 features (181, 30000) (41.43 MB) in 6.3402s
[15.83s] Starting training with 16 workers for 80 models
[16.88s] Trained multirockethydra-bestk-p-ridgecv_s_1055700636 in 1.0174s for f-8 (3.06 MB)
[17.75s] Trained multirockethydra-bestk-p-ridgecv_s_1055700636 in 1.8983s for f-0 (3.06 MB)
[17.81s] Trained multirockethydra-bestk-p-ridgecv_s_1055700636 in 1.9587s for f-1 (3.06 MB)
[17.82s] Trained multirockethydra-bestk-p-ridgecv_s_1055700636 in 1.9698s for f-7 (3.06 MB)
[17.86s] Trained multirockethydra-bestk-p-ridgecv_s_1055700636 in 2.0151s for f-6 (3.06 MB)
[17.88s] Trained multirockethydra-bestk-p-ridgecv_s_105

,random_state,26
,k_folds,10
,n_jobs,16
,keep_features,True
,verbose,10
,model_names,None
,n_repetitions,2


In [7]:
#preds = m.predict_per_model(X_test)
#tests_accs = []
#for model_name, model_preds in preds.items():
#    acc = accuracy_score(y_test, model_preds)
#    tests_accs.append({"model": model_name, "test_accuracy": acc})
#test_df = pl.DataFrame(tests_accs)

In [8]:
#pl.DataFrame(m.summary())

In [9]:
#test_df

In [10]:
#pl.DataFrame(m.summary()).join(test_df, on="model").sort("test_accuracy", descending=True)

In [11]:
y_pred = m.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy on {dataset}: {acc}")

[0.00s] Starting prediction
[1.71s] Computed multirocket_s_292867638 features (77, 49728) (14.61 MB) in 1.6365s
[6.35s] Computed hydra_s_732432917 features torch.Size([77, 7168]) (2.11 MB) in 4.6328s
[10.41s] Computed quant features (77, 7987) (2.35 MB) in 4.0600s
[14.59s] Computed rdst_s_121263262 features (77, 30000) (17.62 MB) in 4.1785s
[14.59s] Computed and saved features for prediction
[14.59s] Starting prediction with 16 workers for 80 first-level models
[53.34s] Predicted rstsf_s_841635173 in 38.6682s
[54.61s] Predicted rstsf_s_841635173 in 39.9605s
[55.07s] Predicted rstsf_s_841635173 in 40.3752s
[55.08s] Predicted rstsf_s_841635173 in 40.4208s
[55.21s] Predicted rstsf_s_841635173 in 40.5360s
[55.27s] Predicted rdst-p-ridgecv_s_503369144 in 0.0474s
[55.34s] Predicted rdst-p-ridgecv_s_503369144 in 0.0557s
[55.42s] Predicted rdst-p-ridgecv_s_503369144 in 0.0742s
[55.50s] Predicted rstsf_s_841635173 in 40.7809s
[55.51s] Predicted rdst-p-ridgecv_s_503369144 in 0.0825s
[55.54s] Pre

In [12]:
#F = m.get_features()
#F.estimated_size('gb')

In [13]:
#P = m.get_oof_predictions()
#P.estimated_size('gb')

In [14]:
proba = m.predict_proba(X_test)
classes = list(m.classes_)

print(f"Log-loss: {log_loss(y_test, proba, labels=classes):.4f}")
print(f"AUC (OvR): {roc_auc_score(y_test, proba, multi_class='ovr', labels=classes):.4f}")

[0.00s] Starting prediction
[1.82s] Computed multirocket_s_292867638 features (77, 49728) (14.61 MB) in 1.7769s
[5.55s] Computed hydra_s_732432917 features torch.Size([77, 7168]) (2.11 MB) in 3.7338s
[8.12s] Computed quant features (77, 7987) (2.35 MB) in 2.5717s
[11.31s] Computed rdst_s_121263262 features (77, 30000) (17.62 MB) in 3.1808s
[11.31s] Computed and saved features for prediction
[11.31s] Starting prediction with 16 workers for 80 first-level models
[49.96s] Predicted rstsf_s_841635173 in 38.6290s
[50.27s] Predicted rstsf_s_841635173 in 38.9294s
[50.37s] Predicted rstsf_s_841635173 in 39.0227s
[50.73s] Predicted rstsf_s_841635173 in 39.3875s
[50.84s] Predicted rstsf_s_841635173 in 39.4997s
[50.91s] Predicted rdst-p-ridgecv_s_503369144 in 0.0695s
[50.96s] Predicted rdst-p-ridgecv_s_503369144 in 0.0390s
[51.00s] Predicted rdst-p-ridgecv_s_503369144 in 0.0366s
[51.07s] Predicted rdst-p-ridgecv_s_503369144 in 0.0552s
[51.12s] Predicted rstsf_s_841635173 in 39.7908s
[51.15s] Pred